# ГП4 — обучение на Kaggle (GPU T4)

Учить **здесь**, не на домашнем ПК.

**Add data — два датасета:**
1. Старый zip ГП2 (val-сегмент).
2. Новый zip шести train-сегментов (`kaggle_gp4_train_segments.zip`).

Internet On, Persistence On, `WANDB_API_KEY`.
Код: git clone `missing-ml-2026`.
Train ~40 эпох, imgsz 640, **1–2 часа**. Eval на **другом** клипе (старый сегмент целиком).


In [ ]:
%pip install -q ultralytics pyarrow opencv-python-headless wandb hydra-core omegaconf pyyaml


Два датасета в Input. `WAYMO_ROOT=/kaggle/input` — скрипты сами найдут parquet по всем zip.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/abbos-trnv/pose_estimation.git"
BRANCH = "missing-ml-2026"
REPO = Path("/kaggle/working/pose_estimation")

def _git(*args):
    subprocess.check_call(["git", *args])

if (REPO / ".git").exists():
    _git("-C", str(REPO), "fetch", "origin", BRANCH)
    _git("-C", str(REPO), "checkout", BRANCH)
    _git("-C", str(REPO), "pull", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO))

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("REPO", REPO, "HEAD")
subprocess.check_call(["git", "log", "-1", "--oneline"])

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print("wandb secret", e)


In [ ]:
!python scripts/export_yolo_pose.py data.max_frames=null


In [ ]:
!python scripts/train_pose.py


In [ ]:
from pathlib import Path
w = Path("runs/train/gp4_multi/weights/best.pt")
print("eval weights", w, "exists", w.exists())
if not w.exists():
    raise FileNotFoundError(w)


In [ ]:
import subprocess
cmd = [
    "python",
    "scripts/eval_pose.py",
    f"model.weights={w}",
    "data=waymo_full",
    "data.subset=all",
    "protocol=both",
]
print(cmd)
subprocess.check_call(cmd)
